# LegalGPT: benchmark the 7B on Colab, exactly as the app runs it

This runs the **same files the laptop app uses**: the official Qwen2.5-7B-Instruct 4-bit GGUF plus
your adapter `lora/v0.7-7b-l4.gguf`, through llama.cpp. On an L4 the whole model fits on the GPU.

**Before you start**
1. Upload the **new** `legalgpt_colab.zip` to `MyDrive/legalgpt/`, replacing the old one.
2. Check that `MyDrive/legalgpt/lora/v0.7-7b-l4.gguf` exists (the training notebook wrote it).
3. **Runtime → Change runtime type → L4 GPU**, then **Runtime → Run all**.

**What it does** (about 1 hour and roughly 5 compute units):
- answers 4 test questions first, and **stops with the error shown** if any fail;
- then answers all 150 questions, saving each answer to Drive as it goes;
- finally **disconnects the GPU**.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, subprocess, shutil, zipfile
DRIVE, REPO = "/content/drive/MyDrive/legalgpt", "/content/legaltech"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print(gpu)
assert "T4" not in gpu, "Change the runtime to L4 (Runtime -> Change runtime type)"
assert os.path.exists(f"{DRIVE}/lora/v0.7-7b-l4.gguf"), "MyDrive/legalgpt/lora/v0.7-7b-l4.gguf is missing"

In [ ]:
# fresh copy of the code (the new zip), and the model files on Colab's own disk (not read from Drive)
shutil.rmtree(REPO, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/legalgpt_colab.zip").extractall(REPO)
os.makedirs("/content/models", exist_ok=True)
shutil.copy(f"{DRIVE}/lora/v0.7-7b-l4.gguf", "/content/models/v0.7-7b-l4.gguf")
os.makedirs(f"{DRIVE}/bench", exist_ok=True)
os.makedirs(f"{REPO}/data/reports/bench", exist_ok=True)
os.symlink(f"{DRIVE}/bench", f"{REPO}/data/reports/bench/baselines")   # answers saved to Drive as they come

In [ ]:
# llama.cpp with CUDA, plus the CUDA 12 runtime it was built for (this Colab image ships CUDA 13)
!pip install -q -r {REPO}/backend/scripts/colab/requirements-colab.txt
!pip install -q --only-binary=:all: llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q "nvidia-cuda-runtime-cu12==12.4.*" "nvidia-cublas-cu12==12.4.*"
import glob, ctypes
cuda12 = sorted({os.path.dirname(p) for p in glob.glob("/usr/local/lib/python3*/dist-packages/nvidia/*/lib/lib*.so.12*")})
os.environ["LD_LIBRARY_PATH"] = ":".join(cuda12 + [os.environ.get("LD_LIBRARY_PATH", "")])  # for the benchmark process
for lib in ("libcudart.so.12", "libcublasLt.so.12", "libcublas.so.12"):  # for this notebook
    ctypes.CDLL(next(p for d in cuda12 for p in glob.glob(f"{d}/{lib}*")), mode=ctypes.RTLD_GLOBAL)
import llama_cpp
print("llama.cpp", llama_cpp.__version__, "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
assert llama_cpp.llama_supports_gpu_offload(), "llama.cpp has no GPU support here"

In [ ]:
# the official 4-bit base model (4.68 GB, two parts), straight from Hugging Face
from huggingface_hub import hf_hub_download
for f in ("qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf", "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf"):
    hf_hub_download("Qwen/Qwen2.5-7B-Instruct-GGUF", f, local_dir="/content/models")
open(f"{REPO}/.env", "w").write(
    "HEAVY_MODEL_GGUF=/content/models/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf\n"
    "HEAVY_LORA_GGUF=/content/models/v0.7-7b-l4.gguf\n"
    "HEAVY_GPU_LAYERS=99\n")   # every layer on the GPU
print(os.listdir("/content/models"))

In [ ]:
def sh(cmd):
    """Run a command, stream its output, and stop the notebook if it fails."""
    env = {**os.environ, "TQDM_MININTERVAL": "60", "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in p.stdout:
        print(line, end=""); out.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode})")
    return "".join(out)

# 1. four test questions first: stop here (before spending more units) if any of them fails
smoke = sh("python backend/scripts/run_baselines.py run --smoke --rewrite --suffix _v0.7-7b --systems legalgpt-heavy")
assert "[FAILED]" not in smoke and smoke.count("[ok]") == 4, "the test questions failed: see the output above"
print("\ntest questions OK")

In [ ]:
# 2. all 150 questions (resumes from the answers already in Drive), then the scores
import time
t0 = time.time()
sh("python backend/scripts/run_baselines.py run --rewrite --suffix _v0.7-7b --systems legalgpt-heavy")
print(f"done in {(time.time() - t0) / 60:.0f} min")
sh("python backend/scripts/run_baselines.py score | grep -E 'system|heavy'")

In [ ]:
# stop the GPU so no more compute units are used (the answers are in Drive)
from google.colab import runtime
runtime.unassign()

## Bring it back
Download `MyDrive/legalgpt/bench/legalgpt-heavy_v0.7-7b.jsonl` into the laptop's
`data/reports/bench/baselines/` folder. Claude then scores it and draws the model-size figure.